In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from src.collect import load_raw

df = load_raw()
print(df.shape)
df.dtypes

C:\Users\Gamer\miniconda3\envs\amazon-reviews\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(1644621, 14)


main_category      object
title              object
average_rating    float64
rating_number       int64
features           object
description        object
price              object
store              object
categories         object
details            object
parent_asin        object
subtitle           object
author             object
category           object
dtype: object

In [2]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "pct": (df.isna().mean() * 100).round(2)
})
missing

,missing,pct
main_category,45532,2.77
title,0,0.00
average_rating,0,0.00
rating_number,0,0.00
features,0,0.00
description,0,0.00
price,0,0.00
store,23696,1.44
categories,0,0.00
details,0,0.00


In [3]:
print("Duplicate parent_asin + category:", df.duplicated(subset=["parent_asin", "category"]).sum())
print("Duplicate parent_asin overall:", df["parent_asin"].duplicated().sum())

cats_per_product = df.groupby("parent_asin")["category"].nunique()
print("Products in more than one category:", (cats_per_product > 1).sum())

Duplicate parent_asin + category: 0
Duplicate parent_asin overall: 0
Products in more than one category: 0


In [4]:
print(df["price"].value_counts(dropna=False).head(30))
print()
print(df["average_rating"].describe())
print()
print(df["rating_number"].describe())

price
None     890371
9.99      21661
19.99     16916
11.99     13964
14.99     13640
8.99      12034
12.99     12017
7.99      11442
15.99     10310
16.99      9997
13.99      9991
29.99      9975
10.99      9119
24.99      8707
6.99       8645
17.99      7780
18.99      7286
39.99      7141
21.99      5623
22.99      5175
5.99       5044
49.99      4912
25.99      4542
23.99      4223
34.99      3877
26.99      3871
20.99      3621
59.99      3229
27.99      3154
19.95      2906
Name: count, dtype: int64

count    1.644621e+06
mean     4.221642e+00
std      7.725122e-01
min      1.000000e+00
25%      4.000000e+00
50%      4.400000e+00
75%      4.700000e+00
max      5.000000e+00
Name: average_rating, dtype: float64

count    1.644621e+06
mean     1.300950e+02
std      8.490285e+02
min      1.000000e+00
25%      4.000000e+00
50%      1.400000e+01
75%      5.400000e+01
max      2.017420e+05
Name: rating_number, dtype: float64


In [5]:
# Is "None" a real missing value or the text "None"?
print("Real missing (NaN/None):", df["price"].isna().sum())
print("Text 'None':", (df["price"] == "None").sum())

# Which non-missing prices can't be turned into numbers?
price_num = pd.to_numeric(df["price"], errors="coerce")
bad = df.loc[price_num.isna() & df["price"].notna() & (df["price"] != "None"), "price"]
print("Unparseable prices:", len(bad))
print(bad.value_counts().head(20))
print()
print(price_num.describe())

Real missing (NaN/None): 0
Text 'None': 890371
Unparseable prices: 60
price
—              37
from 23.99      2
from 70.61      1
from 8.00       1
from 7.51       1
from 1.25       1
from 7.99       1
from 109.99     1
from 12.99      1
from 26.85      1
from 24.00      1
from 19.99      1
from 25.87      1
from 63.36      1
from 1.77       1
from 28.40      1
from 50.85      1
from 24.87      1
from 25.39      1
from 29.99      1
Name: count, dtype: int64

count    754190.000000
mean         53.238520
std         289.677176
min           0.000000
25%          10.990000
50%          19.490000
75%          38.990000
max      129266.640000
Name: price, dtype: float64


In [6]:
   %%writefile src/clean.py
import numpy as np
import pandas as pd

PRICE_MISSING_TEXT = {"None", "—", "-", ""}


def clean_price(s: pd.Series) -> pd.DataFrame:
    """Convert raw price strings to numbers.
    'None' and dashes become NaN, 'from X' keeps X and is flagged,
    and prices <= 0 become NaN."""
    raw = s.astype("string").str.strip()
    is_from = raw.str.startswith("from ", na=False)
    text = raw.str.replace(r"^from\s+", "", regex=True)
    text = text.str.replace(r"[$,]", "", regex=True)
    text = text.mask(text.isin(PRICE_MISSING_TEXT))
    num = pd.to_numeric(text, errors="coerce")
    num = num.mask(num <= 0)
    return pd.DataFrame({
        "price_num": num.astype("float64"),
        "price_is_from": is_from.astype(bool),
        "log_price": np.log1p(num).astype("float64"),
    })

Overwriting src/clean.py


In [7]:
%load_ext autoreload
%autoreload 2
from src.clean import clean_price

print("Prices equal to 0:", (pd.to_numeric(df["price"], errors="coerce") == 0).sum())

price_cols = clean_price(df["price"])
df = df.drop(columns=price_cols.columns, errors="ignore").join(price_cols)

print("Rows:", len(df))
print("Prices usable:", df["price_num"].notna().sum())
print("Prices missing:", df["price_num"].isna().sum())
print("'from' prices:", df["price_is_from"].sum())
df[["price", "price_num", "price_is_from", "log_price"]].sample(10, random_state=0)

Prices equal to 0: 2
Rows: 1644621
Prices usable: 754211
Prices missing: 890410
'from' prices: 23


,price,price_num,price_is_from,log_price
90519,None,NaN,False,NaN
97620,None,NaN,False,NaN
341790,18.0,18.00,False,2.944439
1051833,13.95,13.95,False,2.704711
913876,24.99,24.99,False,3.257712
76089,None,NaN,False,NaN
721222,None,NaN,False,NaN
1065280,None,NaN,False,NaN
260124,None,NaN,False,NaN
136461,None,NaN,False,NaN


In [8]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "pct": (df.isna().mean() * 100).round(2)
})
print(missing)
print()
print("Duplicate parent_asin + category:", df.duplicated(subset=["parent_asin", "category"]).sum())
print("Duplicate parent_asin overall:", df["parent_asin"].duplicated().sum())
cats_per_product = df.groupby("parent_asin")["category"].nunique()
print("Products in more than one category:", (cats_per_product > 1).sum())
print()
print(df["rating_number"].describe())

                missing    pct
main_category     45532   2.77
title                 0   0.00
average_rating        0   0.00
rating_number         0   0.00
features              0   0.00
description           0   0.00
price                 0   0.00
store             23696   1.44
categories            0   0.00
details               0   0.00
parent_asin           0   0.00
subtitle        1644176  99.97
author          1644448  99.99
category              0   0.00
price_num        890410  54.14
price_is_from         0   0.00
log_price        890410  54.14

Duplicate parent_asin + category: 0
Duplicate parent_asin overall: 0
Products in more than one category: 0

count    1.644621e+06
mean     1.300950e+02
std      8.490285e+02
min      1.000000e+00
25%      4.000000e+00
50%      1.400000e+01
75%      5.400000e+01
max      2.017420e+05
Name: rating_number, dtype: float64


In [9]:
%%writefile -a src/clean.py


def _list_len(x):
    try:
        return len(x)
    except TypeError:
        return 0


def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    """Clean the raw product table. Pass a freshly loaded raw frame;
    the raw parquet file on disk is never modified."""
    df = df.drop(columns=["subtitle", "author"])
    for col in ["store", "main_category"]:
        s = df[col].astype("string").str.strip()
        df[col] = s.mask(s == "").fillna("Unknown")
    df = df.join(clean_price(df["price"]))
    df["has_price"] = df["price_num"].notna()
    df["log_rating_number"] = np.log1p(df["rating_number"])
    df["n_features"] = df["features"].map(_list_len)
    df["n_description"] = df["description"].map(_list_len)
    return df

Appending to src/clean.py


In [10]:
from src.clean import clean_products

del df  # free memory before loading a fresh copy
df_clean = clean_products(load_raw())

print("Rows after cleaning:", len(df_clean))
print("Columns:", df_clean.shape[1])
print("Store 'Unknown':", (df_clean["store"] == "Unknown").sum())
print("main_category 'Unknown':", (df_clean["main_category"] == "Unknown").sum())
print("Has price:", df_clean["has_price"].sum())

os.makedirs("data/processed", exist_ok=True)
df_clean.to_parquet("data/processed/products_clean.parquet", index=False)
print("Saved data/processed/products_clean.parquet")

Rows after cleaning: 1644621
Columns: 19
Store 'Unknown': 25688
main_category 'Unknown': 45532
Has price: 754211
Saved data/processed/products_clean.parquet


## Cleaning decisions (Person 2)

| Step | Decision | Rows affected | Rows after |
|---|---|---|---|
| Raw load | 4 categories, 14 columns | — | 1,644,621 |
| Duplicates | 0 duplicate parent_asin; 0 products in multiple categories; nothing removed | 0 | 1,644,621 |
| price | Text "None" (890,371), dashes (37), and $0 (2) set to missing; "from X" (23) keeps X and is flagged in `price_is_from`; no imputation | 890,410 missing | 1,644,621 |
| price | Added `price_num`, `log_price`, `has_price` (754,211 usable prices) | — | 1,644,621 |
| store | Missing or blank filled with "Unknown" | 25,688 | 1,644,621 |
| main_category | Missing filled with "Unknown" (`category` source tag is complete) | 45,532 | 1,644,621 |
| subtitle, author | Dropped (over 99.9% missing) | — | 1,644,621 |
| rating_number | Heavily skewed (median 14, max 201,742); added `log_rating_number` | — | 1,644,621 |
| features, description | Added `n_features`, `n_description` (list lengths) | — | 1,644,621 |
| Output | Saved to `data/processed/products_clean.parquet`; raw file unchanged | — | 1,644,621 rows, 19 columns |